In [ ]:
%matplotlib widget
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.pyplot as plt
import chemiscope

import scwidgets
from scwidgets.check import (
    Check,
    CheckRegistry,
    assert_numpy_allclose,
    assert_numpy_floating_sub_dtype,
    assert_shape,
    assert_type,
)
from scwidgets.answer import AnswerRegistry
from scwidgets.code import ParameterPanel
from scwidgets.cue import CueObject, CueFigure
from scwidgets.exercise import CodeExercise, TextExercise

import ase
from ase.io import read, write

In [ ]:
scwidgets.get_css_style()

In [ ]:
answer_registry = AnswerRegistry(filename_prefix="module_01")
answer_registry

In [ ]:
check_registry = CheckRegistry()
check_registry

In [ ]:
module_summary = TextExercise(
    exercise_description="If you have general comments on this module",
    answer_registry=answer_registry,
    answer_key="Module summary",
)
display(module_summary)

# Atomic structures on a computer

In an atomistic description of matter, a configuration of a structure is entirely determined by

* The chemical nature of the atoms, $a_i$

* Their positions, $\mathbf{r}_i$, corresponding to a list of $(x,y,z)$ Cartesian coordinates

* Possibly, three unit-cell vectors $\mathbf{h}_{i=1,2,3}$ that descibe the periodicity of a lattice

There is a [babel of formats](http://openbabel.org/), often poorly standardized, that have been developed to store atomic configurations. Notable examples are `pdb` files, used for biological structures (e.g. in the [protein data bank](https://www.rcsb.org/)), `cif` files that are often used to store crystallographic data (the main format for the [Cambridge structural database](https://www.ccdc.cam.ac.uk/solutions/csd-core/components/csd/) and the `xyz` format, one of the simplest (and most abused) formats, in which atomic positions are stored according to the schema

```
N_ATOMS
comment line
Element X Y Z
...
```

The comment line is often abused to add further information, e.g. the lattice parameters following the format 

```
Lattice='h1x h1y h1z h2x h2y h2z h3x h3y h3z'
```

Multiple blocks corresponding to different structures can be simply concatenated -- although many programs assume all structures in a single file to have the same number and type of atoms. 

An example of the content of an `xyz` file (the QM7b dataset, from [DOI: 10.1088/1367-2630/15/9/095003](https://doi.org/10.1088/1367-2630/15/9/095003) ):

```
!head -n 17 data/qm7b-ase.xyz
```

In [ ]:
!head -n 17 data/qm7b-ase.xyz

In [ ]:
ex01_description = """
What is the chemical formula of these two structures? Can you also guess what
actual molecules they correspond to? If you can't figure it out by looking at
the coordinates (which are in Å), we'll look at this file later on, so you can
"cheat" if you can't see the structure based on the distance!
"""
ex01_answer = "Enter your answer here"

### BEGIN SOLUTION
ex01_answer = """The two molecules are CH4 and C2H6. Looking at the interatomic distances, the
first structure has 4 H atoms bound to C, so it's methane. The second has two C
at about 1.5A, and two groups of 3H, each bound to one of the C, so it is
ethane."""
### END SOLUTION


ex01_exercise = TextExercise(
    ex01_answer,
    exercise_description=ex01_description,
    answer_registry=answer_registry,
    answer_name="Exercise 01 - What are the two structures?",
    answer_key="ex01",
)
display(ex01_exercise)

# Loading and defining atomic structures with ASE

In this course we will use the [Atomic Simulation Environment](https://wiki.fysik.dtu.dk/ase/) to load and manipulate structures. ASE stores structures in an `Atoms` class, which contains `positions`, `symbols` and `cell` members. Atomic positions are typically interpreted to be expressed in Ångstrom ($10^{-10}$m).

Structures can be loaded from disk using the `read` command (from `ase.io`)

```
# the second argument determines the slice of the file that will be read (e.g. 0 to load the first frame)
# it can be either a python slice() or a string representation with the usual start[:end][:stride] format
qm7 = read("data/qm7b-ase.xyz", ":")  
```

or created manually

```
methane = ase.Atoms(symbols="CH4", positions=[ 
    [1.00, -0.00, -0.00], 
    [2.09, 0.00,  0.00], 
    [0.63,  1.03,  0.00], 
    [0.63, -0.53,  0.88],
    [0.64, -0.51, -0.91]]
    )
```

The atomic positions, labels, or the unit cell can also be modified as common arrays

```
methane.symbols[1] = "Cl"   # turn the molecule into chloromethane
```

*NB:*
1. the frame indices are 0-based
2. atoms indices are 0-based
3. symbols, positions and cell can be manipulated as arrays, but implement some syntactic sugar, e.g. you can set symbols in compact, string form, e.g. `methane.symbols = "CH4"`
See the [documentation for the `ase.Atoms` object](https://wiki.fysik.dtu.dk/ase/ase/atoms.html) for more details. 

In [ ]:
def methylammonium():
    """
    Loads the structure #1 from the data/qm7b-ase.xyz file,
    and modifies the composition so that it corresponds to CH3NH3+.

    :return: an ASE atoms object that describes the molecular structure
    """
    # Write your solution, then click on the button below to update the plotter
    # and check against the reference value

    import ase
    from ase.io import read

    structure = []  # load here

    ### BEGIN SOLUTION
    structure = read("data/qm7b-ase.xyz", 1)
    structure.symbols[1] = "N"
    ### END SOLUTION

    return structure


def ex02_asserts(code_input_outputs: tuple) -> str:
    structure = code_input_outputs[0]
    if not (isinstance(structure, ase.Atoms)):
        return f"Expected type ase.Atoms but got {type(structure)!r}."
    if len(structure) != 8:
        return f"Expected length {8} but got {len(structure)}."
    if np.sum(7 != structure.numbers) == 1:
        return f"One nitrogen is expected in the molecule. Found {np.sum(7 == structure.numbers)} nitrogen(s)."
    if np.sum(1 != structure.numbers) == 6:
        return f"Six hydrogens are expected in the molecule. Found {np.sum(1 == structure.numbers)} hydrogen(s)."
    return ""


def fingerprint_ase_atoms(atoms):
    if not (isinstance(atoms, ase.Atoms)):
        return f"Expected type ase.Atoms but got {type(atoms)!r}."
    return np.sum(atoms.get_all_distances() * atoms.numbers)


def ex02_update(code_exercise):
    structure = code_exercise.run_code()
    if not (isinstance(structure, ase.Atoms)):
        raise ValueError(f"Output should be ase.Atoms not {type(structure)!r}")
    structure = [structure]

    cue_output = code_exercise.cue_outputs[0]
    cue_output.display_object = chemiscope.show(
        frames=structure,
        properties=chemiscope.extract_properties(structure),
        mode="structure",
    )


ex02_description = """
Write a function that loads the structure with index 1 from the
<code>data/qm7b-ase.xyz</code>.  What is it? Modify the structure so it
corresponds tomethylammonium, $\\mathrm{CH_3NH_3^+}$, one of the organic cations used in
<ahref="https://en.wikipedia.org/wiki/Methylammonium_lead_halide">hybrid perovskite solar cells</a>.
Get a nice snapshot of the structure!
"""

ex02_exercise = CodeExercise(
    code=methylammonium,
    check_registry=check_registry,
    cue_outputs=CueObject(),
    update_func=ex02_update,
    answer_key="ex02",
    answer_registry=answer_registry,
    exercise_title="Exercise 02 - Load the two structures",
    exercise_description=ex02_description,
)

check_registry.add_check(
    ex02_exercise,
    asserts=[
        ex02_asserts,
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[{}],
    outputs_references=[(237.7334114111937,)],
    fingerprint=fingerprint_ase_atoms,
)

display(ex02_exercise)

# Unit cell and periodic structures

The ASE format and the structure viewer allows also to manipulate periodic structures corresponding to bulk materials. To do so, you need to set the `cell` member of an `Atoms` structure to contain the (row-major) cell matrix. ASE considers separately the information on whether the unit cell should be considered as indicating just a finite volume that contains the atoms, or as a periodic repeat unit: this is controlled by the `pbc` parameter - standing for Periodic Boundary Conditions.

Polonium is the only element that crystallizes in a simple-cubic structure. It has a density of 9.196 g/cm<sup>3</sup>. The isotope of polonium that can be isolated from uranium ores is $\mathrm{^{210} Po}$,     that has a molar mass of 210 g/mol. Consider that one mole contains `6.02214076e23` atoms.

In [ ]:
ex03a_description = """What is the lattice parameter of simple-cubic Po?"""
ex03a_answer = "Enter your answer here"

### BEGIN SOLUTION
ex03a_answer = """The structure has a lattice parameter of 3.359Å"""
### END SOLUTION

ex03a_exercise = TextExercise(
    ex03a_answer,
    exercise_description=ex03a_description,
    exercise_title="Exercise 03a - Lattice parameter of Po",
    answer_registry=answer_registry,
    answer_key="ex03a",
)
display(ex03a_exercise)

In [ ]:
def polonium():
    """
    Build a unit cell of simple-cubic, alpha-Po.

    :return: an ASE atoms object that describes the unit cell structure
    """
    import ase
    from ase.io import read

    # Write your solution, then click on the button below to update the plotter
    # and check against the reference value
    a0 = 0.0  # lattice parameter

    # complete the call, substituting placeholders with actual values
    # structure = ase.Atoms(symbols="...", positions= ... ,
    #                  cell= [ [...], ...] ,
    #                  pbc=True)

    ### BEGIN SOLUTION
    a0 = ((210 / 6.02214076e23) / (9.196e-24)) ** (1 / 3)
    # complete the call, substituting placeholders with actual values
    structure = ase.Atoms(
        symbols="Po", positions=[[0, 0, 0]], cell=[a0, a0, a0], pbc=True
    )
    ### END SOLUTION

    return structure


def ex03b_asserts(code_input_outputs: tuple) -> str:
    structure = code_input_outputs[0]
    if not (isinstance(structure, ase.Atoms)):
        return f"Expected type ase.Atoms but got {type(structure)!r}."
    if len(structure) != 1:
        return f"Expected length 1 but got {len(structure)}."
    if np.sum(84 != structure.numbers) == 1:
        return f"One Polonium is expected in the molecule. Found {np.sum(84 == structure.numbers)} Polonium atom(s)."
    if not (
        np.allclose(structure.cell.array, 3.3596173 * np.eye(3), atol=1e-2, rtol=1e-1)
    ):
        return f"Unit cell has wrong dimensions."
    return ""


def ex03b_updater(code_exercise):
    structure = code_exercise.run_code()
    if not (isinstance(structure, ase.Atoms)):
        raise ValueError(f"Output should be ase.Atoms not {type(structure)!r}")
    structure = [structure]

    cue_output = code_exercise.cue_outputs[0]
    cue_output.display_object = chemiscope.show(
        frames=structure,
        mode="structure",
        properties={"dummy": {"target": "structure", "values": [0]}},
    )


ex03b_description = """
Write a function that returns an `Atoms` object that describes a single unit cell of Po,
with one atom at the origin, and take a snapshot!
<br><br>
Take this opportunity to experiment with the visualization options for crystalline structures:
by clicking on the ☰ icon, you can choose to visualize the unit cell, and replicate the cell
multiple times along the three axes. Get a nice snapshot of the structure!
"""

ex03b_exercise = CodeExercise(
    code=polonium,
    check_registry=check_registry,
    cue_outputs=CueObject(),
    update_func=ex03b_updater,
    answer_key="ex03b",
    answer_registry=answer_registry,
    exercise_title="Exercise 03b - Single unit cell of Po",
    exercise_description=ex03b_description,
)

check_registry.add_check(
    ex03b_exercise,
    asserts=[
        ex03b_asserts,
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[{}],
    outputs_references=[(0.0,)],
    fingerprint=fingerprint_ase_atoms,
)

display(ex03b_exercise)

# Periodic-boundary conditions, supercells, wrapping structures

Periodic boundary conditions are not only used to model perfect crystals. They are also used as a practical way to describe bulk systems, while using a finite number of atomic degrees of freedom: the size of the cell and the coordinates of the atoms in a single repeate unit. Compare these two systems:

a) a finite-sized droplet with 10 water molecules
<img src="figures/pbc-1.png" width="400"/>
b) a periodic system with a repeat unit of 10 water molecules
<img src="figures/pbc-2.png" width="400"/>

In [ ]:
ex04_description = """
Discuss briefly in which ways the two scenarios differ from bulk water: these
are usually referred to as <em>finite-size effects.</em>  You can think of the
impact of having just a finite number of water molecules in terms of the atomic
environment "seen" by each water molecule, or discuss in more macroscopic terms
based on bulk and interfaces. Which of the two cases would you expect to be
closer to the limit of a large number of water molecules?
"""
ex04_answer = ""

### BEGIN SOLUTION
ex04_answer = """
The periodic supercell approach is likely to approximate better a bulk system,
because molecules at the edge of the supercell are entirely surrounded by other
molecules, while the finite structure will have a boundary with a liquid/vacuum
interface.
"""
### END SOLUTION

ex04_exercise = TextExercise(
    ex04_answer,
    exercise_description=ex04_description,
    exercise_title="Exercise 04 - Finite-size effects",
    answer_registry=answer_registry,
    answer_key="ex04",
)
display(ex04_exercise)

We will see later how we can define and compute interactions in a periodic system such as this. For the moment, let's look at a snapshot from a real simulation of liquid water, with a supercell containing 32 water molecules. You'll need to load the file `data/h2o-32-snapshot.xyz`, and return it as an `Atoms` object. Switch on the unit cell visualization and look at the position of the water molecules relative to it.

In [ ]:
def water_pbc():
    """
    Loads data/h2o-32-snapshot.xyz as an ase.Atoms object, folds the atomic
    coordinates into the supercell, and returns it so it can be visualized

    :return: an ASE atoms object containing the atomic coordinates "folded"
         into the unit cell.
    """
    # Write your solution, then click on the button below to update the plotter
    # and check against the reference value

    import ase
    from ase.io import read

    # complete the call, substituting placeholders with actual values
    # structure = read( ... )

    # add here code to wrap the structure. you'll need to use the cell parameters and the
    # atomic positions.
    # NB1: ASE has a wrap() method
    # structure.wrap()
    # you can use it to get an idea of what should happen, but you'll have to implement
    # wrapping yourself
    # NB2: the positions of each atom should be modified, shifting them by an integer number
    #      of lattice parameters so they are between 0 and the lattice parameter.
    # NB3: you can (and should) exploit the fact that the supercell is cubic, but if you
    #     want extra brownie points you can try a general version.

    # structure.positions = ....

    ### BEGIN SOLUTION
    structure = read("data/h2o-32-snapshot.xyz", 0)
    a0 = structure.cell[0, 0]
    for i in range(len(structure.positions)):
        for j in range(3):
            while structure.positions[i, j] > a0:
                structure.positions[i, j] -= a0
            while structure.positions[i, j] < 0:
                structure.positions[i, j] += a0
    ### END SOLUTION
    return structure


def ex05_updater(code_exercise):
    structure = code_exercise.run_code()
    if not (isinstance(structure, ase.Atoms)):
        raise ValueError(f"Output should be ase.Atoms not {type(structure)!r}")
    structure = [structure]

    cue_output = code_exercise.cue_outputs[0]
    cue_output.display_object = chemiscope.show(
        frames=structure,
        mode="structure",
        properties={"dummy": {"target": "structure", "values": [0]}},
    )


def ex05_asserts(code_input_outputs: tuple) -> str:
    structure = code_input_outputs[0]
    if not (isinstance(structure, ase.Atoms)):
        return f"Expected type ase.Atoms but got {type(structure)!r}."
    if len(structure) != 96:
        return f"Expected length 96 but got {len(structure)}."
    return ""


ex05_description = """
After you have looked at the original structure, write code to "fold" the
coordinates of the atoms so that they are within the unit cell. Some molecules
will be "broken" across cell boundaries - you can see what happens when you
visualize multiple periodic replicas, using the `supercell` options in the
visualizer.
"""

ex05_exercise = CodeExercise(
    code=water_pbc,
    check_registry=check_registry,
    cue_outputs=CueObject(),
    update_func=ex05_updater,
    answer_key="ex05",
    answer_registry=answer_registry,
    exercise_title="Exercise 05 - Supercell",
    exercise_description=ex05_description,
)

check_registry.add_check(
    ex05_exercise,
    asserts=[
        ex05_asserts,
        assert_type,
        assert_shape,
        assert_numpy_allclose,
    ],
    inputs_parameters=[{}],
    outputs_references=[(199303.2380212066,)],
    fingerprint=fingerprint_ase_atoms,
)

display(ex05_exercise)

In [ ]:
ex05_description = """
Does the structure before folding "look" at all like liquid water? Do you think
that the physical properties of the structure with the atoms folded within the
unit cell are different from those of the original structure? Keep in mind that
a supercell is meant to describe an infinite bulk solid, and not only the set
of atoms that are explicitly displayed.
"""
ex05_answer = "Enter your answer here"

### BEGIN SOLUTION
ex05_answer = """
The structure before and after folding represents the same periodic structure:
once the initial structure is replicated to infinity, the atoms outside the
unit cell will interact directly with the periodic replicas in other cells.
Similarly, the molecules that are broken across the periodic boundaries after
are actually intact, because of the presence of atoms in the nearby cells.
"""
### END SOLUTION

ex05_exercise = TextExercise(
    ex05_answer,
    exercise_description=ex05_description,
    exercise_title="Exercise 05",
    answer_registry=answer_registry,
    answer_key="ex05",
)
display(ex05_exercise)